# Multi-Task Surgical Workflow Analysis
## Joint Phase Recognition and Tool Tracking in Laparoscopic Surgery

**Student:** Omar Morsi (40236376)

**Course:** COMP 432 -- Machine Learning

---

### Abstract

This project develops a deep learning system to automatically recognize surgical phases and track instruments in laparoscopic cholecystectomy videos. Using the Cholec80 dataset (80 videos), the system performs two simultaneous tasks: classifying the current surgical stage and detecting the presence of seven surgical tools. The core architecture combines a ResNet-50 CNN for spatial feature extraction with temporal models (LSTM, Multi-Stage TCN) to capture the chronological sequence of the procedure. A novel **Correlation Loss** exploits the statistical relationship between tool presence and surgical phase to reduce logically inconsistent predictions. We compare four model variants and evaluate using frame-wise F1-score, mean Average Precision, and segment-level edit score.

## 1. Introduction

Laparoscopic surgery is a minimally invasive technique widely used for procedures like cholecystectomy (gallbladder removal). While it offers benefits such as smaller incisions and faster recovery, it introduces challenges for surgical workflow monitoring. Understanding the progression of a surgery in real time is valuable for:

- **Patient safety:** Detecting deviations from standard procedures and alerting the surgical team.
- **Training and feedback:** Providing objective, data-driven assessments of trainee surgeons.
- **Hospital efficiency:** Predicting remaining surgery time for better operating room scheduling.

Currently, analyzing surgical videos is a manual, labor-intensive process prone to human error. Automatic recognition of surgical phases and tool usage can provide real-time decision support and pave the way for "smart" operating rooms.

### Goals

1. Develop a multi-task model that jointly performs **phase recognition** (7 phases) and **tool detection** (7 tools).
2. Compare multiple temporal modeling approaches: no temporal context (baseline), LSTM, and Multi-Stage TCN.
3. Introduce a novel **Correlation Loss** that penalizes impossible tool-phase combinations.
4. Evaluate using standard metrics: frame-wise F1-score, mAP, and edit score for temporal consistency.

## 2. Related Work

Surgical workflow analysis has evolved significantly over the past decade:

- **Hidden Markov Models (HMMs):** Early approaches modeled phase transitions using HMMs with hand-crafted features. While interpretable, they struggled with the visual complexity of surgical videos.

- **EndoNet (Twinanda et al., 2016):** Introduced a CNN-based approach using AlexNet for joint tool and phase recognition. This was the first deep learning method applied to the Cholec80 dataset and established the standard benchmark.

- **SV-RCNet (Jin et al., 2018):** Improved upon EndoNet by adding an LSTM layer on top of ResNet features, enabling the model to capture temporal dependencies between frames.

- **TeCNO / MS-TCN (Czempiel et al., 2020):** Introduced Multi-Stage Temporal Convolutional Networks, which use dilated convolutions to capture long-range temporal dependencies. MS-TCN showed superior performance in reducing phase prediction "flickering" compared to LSTMs.

### How This Work Differs

Most prior work treats tool detection and phase recognition as either isolated tasks or loosely coupled through shared feature extraction. Our approach introduces a **Multi-Task Learning (MTL) framework with a novel Correlation Loss** that explicitly models the statistical relationship between tools and phases. For example, a clipper is strongly associated with the "Clipping and Cutting" phase, and our loss function penalizes predictions that violate these learned co-occurrence patterns.

## 3. Dataset

### 3.1 Cholec80

The Cholec80 dataset (Twinanda et al., 2016) consists of **80 high-definition videos** of cholecystectomy procedures performed at the University Hospital of Strasbourg, France.

- **Phase annotations:** 7 surgical phases per frame
  - Preparation, Calot Triangle Dissection, Clipping and Cutting, Gallbladder Dissection, Gallbladder Packaging, Cleaning and Coagulation, Gallbladder Retraction
- **Tool annotations:** 7 binary tool presence labels per frame
  - Grasper, Bipolar, Hook, Scissors, Clipper, Irrigator, SpecimenBag
- **Total frames:** ~370,000 at 25 fps
- **Subsampled:** to 1 fps (~15,000 frames total) for computational feasibility

### 3.2 Data Split

| Split | Videos | Count | Purpose |
|---|---|---|---|
| Train | 1-32 | 32 | Model training |
| Validation | 33-40 | 8 | Hyperparameter tuning, early stopping |
| Test | 41-80 | 40 | Final evaluation only |

This follows the standard Cholec80 convention (first 40 for training, last 40 for testing). The validation set is carved from the training videos to ensure **no data leakage**. Hyperparameter tuning is performed exclusively on the validation set.

### 3.3 Preprocessing

- Frames resized to 224x224 pixels
- Normalized using ImageNet statistics (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
- **Training augmentations:** random horizontal flip, color jitter, random rotation (10 deg), Gaussian blur (simulates surgical smoke)
- **Validation/Test:** resize and normalize only

In [ ]:
# ============================================================
# Setup: Install dependencies and clone project repository
# ============================================================

# Uncomment the line below to install dependencies (first run only)
# !pip install torch torchvision numpy opencv-python scikit-learn matplotlib seaborn pyyaml tqdm

# If running on Colab, clone the project repo:
# !git clone https://github.com/omorsi45/surgical-workflow-analysis.git
# %cd surgical-workflow-analysis

import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for imports
# Adjust this path based on your setup
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import set_seed, load_config, compute_class_weights
from src.dataset import (
    Cholec80VideoDataset, Cholec80FeatureDataset,
    collate_sequences, get_train_transforms, get_eval_transforms,
    PHASE_NAMES, TOOL_NAMES,
)
from src.models.backbone import ResNet50FeatureExtractor
from src.models.temporal import BaselineModel, LSTMModel, MultiStageTCN
from src.models.multitask import (
    MultiTaskModel, MultiTaskLoss, build_cooccurrence_matrix,
)
from src.train import Trainer
from src.evaluate import (
    compute_phase_f1, compute_per_phase_f1, compute_phase_accuracy,
    compute_tool_map, compute_per_tool_ap, compute_edit_score,
    plot_training_curves, plot_confusion_matrix,
    plot_per_class_metrics, plot_timeline_ribbon,
    plot_cooccurrence_heatmap, plot_model_comparison,
)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Google Colab: Mount Drive and set data path
# Run this cell only on Colab. Skip locally.
# ============================================================

import os

# Mount Google Drive (will prompt for authorization on first run)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Set PROJECT_ROOT to the cloned repo on Colab
    PROJECT_ROOT = "/content/surgical-workflow-analysis"

    # Point data_dir at the Cholec80 folder on your Google Drive.
    # Adjust the path below to match where you uploaded the dataset.
    DRIVE_DATA_DIR = "/content/drive/MyDrive/cholec80"

    # Symlink so the repo-relative path "data" resolves to Drive
    data_link = os.path.join(PROJECT_ROOT, "data")
    if not os.path.exists(data_link):
        os.symlink(DRIVE_DATA_DIR, data_link)
    print(f"Data linked: {data_link} -> {DRIVE_DATA_DIR}")
else:
    print("Running locally — skipping Drive mount.")


In [ ]:
# ============================================================
# Load configuration and set seeds for reproducibility
# ============================================================

config = load_config(os.path.join(PROJECT_ROOT, "configs", "default.yaml"))
set_seed(config["training"]["seed"])

print("Configuration loaded:")
print(f"  Train videos: {len(config['data']['train_videos'])} videos")
print(f"  Val videos:   {len(config['data']['val_videos'])} videos")
print(f"  Test videos:  {len(config['data']['test_videos'])} videos")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Epochs: {config['training']['num_epochs']}")
print(f"  Early stopping patience: {config['training']['early_stopping_patience']}")

### 3.4 Dataset Statistics

Let us examine the class distribution to understand the class imbalance challenge.

In [ ]:
# ============================================================
# Dataset statistics: class distribution analysis
# ============================================================

features_dir = os.path.join(PROJECT_ROOT, config["data"]["features_dir"])

# Collect all training labels
all_train_phases = []
all_train_tools = []
for vid in config["data"]["train_videos"]:
    data = torch.load(
        os.path.join(features_dir, f"video{vid:02d}.pt"), weights_only=True
    )
    all_train_phases.append(data["phases"])
    all_train_tools.append(data["tools"])

all_train_phases = torch.cat(all_train_phases)
all_train_tools = torch.cat(all_train_tools)

# Phase distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

phase_counts = np.bincount(all_train_phases.numpy().astype(int), minlength=7)
colors = sns.color_palette("viridis", 7)
ax1.barh(PHASE_NAMES, phase_counts, color=colors)
ax1.set_xlabel("Number of Frames")
ax1.set_title("Phase Distribution (Training Set)")
for i, v in enumerate(phase_counts):
    ax1.text(v + 50, i, f"{v} ({v/phase_counts.sum()*100:.1f}%)", va="center")

# Tool distribution
tool_counts = all_train_tools.numpy().sum(axis=0)
colors_t = sns.color_palette("magma", 7)
ax2.barh(TOOL_NAMES, tool_counts, color=colors_t)
ax2.set_xlabel("Number of Frames Present")
ax2.set_title("Tool Presence (Training Set)")
for i, v in enumerate(tool_counts):
    ax2.text(v + 50, i, f"{int(v)}", va="center")

plt.tight_layout()
plt.show()

print(f"\nTotal training frames: {len(all_train_phases)}")
print(f"Class imbalance ratio (max/min phase): {phase_counts.max()/phase_counts[phase_counts>0].min():.1f}x")

## 4. Methodology

### 4.1 Architecture Overview

The system uses a two-phase pipeline:

1. **Feature Extraction (offline):** A pretrained ResNet-50 (ImageNet) extracts a 2048-dim feature vector per frame. These are cached to disk.
2. **Temporal Modeling:** A temporal model processes the feature sequence and outputs predictions through two task-specific heads.

```
Video Frame --> [ResNet-50 (frozen)] --> 2048-dim features (cached)
                                              |
                                    [Temporal Model]
                                    /             \
                          [Phase Head]      [Tool Head]
                          7-class softmax   7-class sigmoid
```

### 4.2 Temporal Model Variants

We compare four models to isolate the contribution of each component:

| Model | Architecture | Purpose |
|---|---|---|
| **Baseline** | FC(2048 -> 512) + ReLU + Dropout | No temporal context |
| **LSTM** | 2-layer BiLSTM(512) + projection | Recurrent temporal modeling |
| **MS-TCN** | 4-stage TCN (10 layers/stage, 64 channels) | Dilated convolution temporal modeling |
| **MS-TCN + Corr. Loss** | Same as MS-TCN + correlation loss | Novel contribution |

### 4.3 Loss Functions

The total loss is a weighted sum of three components:

$$L = \lambda_{phase} \cdot L_{phase} + \lambda_{tool} \cdot L_{tool} + \lambda_{corr} \cdot L_{corr}$$

- **Phase loss ($L_{phase}$):** Weighted Cross-Entropy, with weights inversely proportional to phase frequency, to handle class imbalance.
- **Tool loss ($L_{tool}$):** Binary Cross-Entropy with Logits, for multi-label tool detection.
- **Correlation loss ($L_{corr}$):** Our novel contribution.

### 4.4 Correlation Loss (Novel Contribution)

We build a co-occurrence matrix $P(\text{tool}_j | \text{phase}_i)$ from training labels. At training time, for each frame:

1. Get predicted phase $\hat{p} = \arg\max(\text{phase\_logits})$
2. Get tool probabilities $\hat{t} = \sigma(\text{tool\_logits})$
3. Look up prior $P(\text{tool} | \hat{p})$ from the co-occurrence matrix
4. Compute penalty: $L_{corr} = \text{mean}(\hat{t} \cdot (1 - P(\text{tool} | \hat{p})))$

This penalizes high tool predictions when the co-occurrence prior says that tool is rare for the predicted phase (e.g., predicting SpecimenBag during Preparation).

## 5. Experimental Setup

### 5.1 Feature Extraction

We extract ResNet-50 features offline for all 80 videos and cache them as `.pt` files. This is done once and avoids re-running the CNN during temporal model training.

In [ ]:
# ============================================================
# Phase 1: Feature extraction with ResNet-50
# Run this cell ONCE to extract and cache features.
# If features are already cached, this cell will skip extraction.
# ============================================================

features_dir = os.path.join(PROJECT_ROOT, config["data"]["features_dir"])
data_dir = os.path.join(PROJECT_ROOT, config["data"]["data_dir"])

all_video_ids = (
    config["data"]["train_videos"]
    + config["data"]["val_videos"]
    + config["data"]["test_videos"]
)

# Check if features already exist
existing = [vid for vid in all_video_ids
            if os.path.exists(os.path.join(features_dir, f"video{vid:02d}.pt"))]
to_extract = [vid for vid in all_video_ids if vid not in existing]

print(f"Features cached: {len(existing)}/{len(all_video_ids)} videos")

if to_extract:
    print(f"Extracting features for {len(to_extract)} videos...")
    extractor = ResNet50FeatureExtractor(pretrained=True)
    transform = get_eval_transforms(config["data"]["frame_size"])

    skipped = []
    for vid in to_extract:
        dataset = Cholec80VideoDataset(
            data_dir, video_id=vid,
            fps=config["data"]["fps"], transform=transform
        )
        if len(dataset) == 0:
            print(f"  WARNING: video{vid:02d} has no frames — MP4 missing. Skipping.")
            skipped.append(vid)
            continue
        extractor.extract_and_cache(
            dataset, features_dir, video_id=vid,
            batch_size=32, device=device
        )

    if skipped:
        print(f"
Skipped {len(skipped)} videos due to missing MP4s: {skipped}")
        print("Re-download these videos and re-run this cell to complete extraction.")
    print("Feature extraction complete!")
else:
    print("All features already cached. Skipping extraction.")

### 5.2 Training Configuration

| Parameter | Value | Rationale |
|---|---|---|
| Optimizer | Adam | Standard for deep learning |
| Learning rate | 1e-4 | Stable convergence for temporal models |
| Weight decay | 1e-5 | L2 regularization |
| Scheduler | Cosine Annealing (T0=10) | Warm restarts help escape local minima |
| Gradient clipping | 5.0 | Prevents exploding gradients in LSTM/TCN |
| Early stopping | Patience=15 on val F1 | Prevents overfitting |
| Batch size | 1 (full video) | Variable-length sequences |
| Seed | 42 | Reproducibility |

In [ ]:
# ============================================================
# Prepare data loaders and class weights
# ============================================================

from torch.utils.data import DataLoader

# Create datasets
train_dataset = Cholec80FeatureDataset(features_dir, config["data"]["train_videos"])
val_dataset = Cholec80FeatureDataset(features_dir, config["data"]["val_videos"])
test_dataset = Cholec80FeatureDataset(features_dir, config["data"]["test_videos"])

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config["training"]["batch_size"],
    shuffle=True, collate_fn=collate_sequences
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False, collate_fn=collate_sequences
)
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False, collate_fn=collate_sequences
)

# Compute class weights for phase loss (handles imbalance)
phase_weights = compute_class_weights(
    all_train_phases.numpy(), config["data"]["num_phases"]
).to(device)
print(f"Phase class weights: {phase_weights}")

# Build co-occurrence matrix for correlation loss
cooccur_matrix = build_cooccurrence_matrix(
    features_dir, config["data"]["train_videos"]
)
print(f"\nCo-occurrence matrix P(tool | phase):")
print(cooccur_matrix.numpy().round(2))

In [ ]:
# ============================================================
# Helper function: train and evaluate a model variant
# ============================================================

def train_model(model_name, temporal_model, use_corr_loss=False):
    """Train a multi-task model variant and return results.

    Args:
        model_name (str): Name for logging and saving.
        temporal_model (nn.Module): Temporal model instance.
        use_corr_loss (bool): Whether to use correlation loss.

    Returns:
        tuple: (model, history) -- trained model and training history.
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    set_seed(config["training"]["seed"])

    model = MultiTaskModel(
        temporal_model,
        num_phases=config["data"]["num_phases"],
        num_tools=config["data"]["num_tools"],
    )

    loss_fn = MultiTaskLoss(
        phase_weights=phase_weights,
        cooccurrence_matrix=cooccur_matrix if use_corr_loss else None,
        lambda_phase=config["training"]["lambda_phase"],
        lambda_tool=config["training"]["lambda_tool"],
        lambda_corr=config["training"]["lambda_corr"] if use_corr_loss else 0.0,
    )

    save_dir = os.path.join(PROJECT_ROOT, "checkpoints", model_name)
    trainer = Trainer(
        model, loss_fn, train_loader, val_loader,
        config, save_dir=save_dir, device=device
    )

    history = trainer.train()
    print(f"\nBest validation F1: {trainer.best_val_f1:.4f}")

    return model, history

### 5.3 Model Training

We train all four model variants sequentially. Each uses the same data splits, class weights, and random seed for fair comparison.

In [ ]:
# ============================================================
# Model 1: Baseline (no temporal modeling)
# ============================================================

baseline_temporal = BaselineModel(
    feature_dim=config["model"]["feature_dim"],
    hidden_dim=config["model"]["hidden_dim"],
    dropout=config["model"]["dropout"],
)
baseline_model, baseline_history = train_model("baseline", baseline_temporal)

In [ ]:
# ============================================================
# Model 2: LSTM
# ============================================================

lstm_temporal = LSTMModel(
    feature_dim=config["model"]["feature_dim"],
    hidden_dim=config["model"]["hidden_dim"],
    num_layers=config["model"]["lstm_layers"],
    bidirectional=config["model"]["lstm_bidirectional"],
    dropout=config["model"]["dropout"],
)
lstm_model, lstm_history = train_model("lstm", lstm_temporal)

In [ ]:
# ============================================================
# Model 3: MS-TCN (without correlation loss)
# ============================================================

tcn_temporal = MultiStageTCN(
    feature_dim=config["model"]["feature_dim"],
    hidden_dim=config["model"]["hidden_dim"],
    num_stages=config["model"]["tcn_stages"],
    num_layers=config["model"]["tcn_layers_per_stage"],
    channels=config["model"]["tcn_channels"],
    dropout=config["model"]["dropout"],
)
tcn_model, tcn_history = train_model("ms_tcn", tcn_temporal)

In [ ]:
# ============================================================
# Model 4: MS-TCN + Correlation Loss (our novel contribution)
# ============================================================

tcn_corr_temporal = MultiStageTCN(
    feature_dim=config["model"]["feature_dim"],
    hidden_dim=config["model"]["hidden_dim"],
    num_stages=config["model"]["tcn_stages"],
    num_layers=config["model"]["tcn_layers_per_stage"],
    channels=config["model"]["tcn_channels"],
    dropout=config["model"]["dropout"],
)
tcn_corr_model, tcn_corr_history = train_model(
    "ms_tcn_corr", tcn_corr_temporal, use_corr_loss=True
)

## 6. Results

### 6.1 Training Curves

In [ ]:
# ============================================================
# Plot training curves for all models
# ============================================================

all_histories = [baseline_history, lstm_history, tcn_history, tcn_corr_history]
all_names = ["Baseline", "LSTM", "MS-TCN", "MS-TCN + Corr."]

fig = plot_training_curves(all_histories, all_names)
plt.show()

In [ ]:
# ============================================================
# Evaluate all models on test set
# ============================================================

@torch.no_grad()
def evaluate_model(model, test_loader, device):
    """Evaluate a model on the test set and return all predictions.

    Args:
        model (nn.Module): Trained MultiTaskModel.
        test_loader (DataLoader): Test data loader.
        device (str): Device string.

    Returns:
        dict: Contains per-video and aggregated predictions/targets.
    """
    model.eval()
    model.to(device)

    all_phase_preds = []
    all_phase_targets = []
    all_tool_preds = []
    all_tool_targets = []
    per_video_results = []

    for features, phases, tools, mask in test_loader:
        features = features.to(device)
        mask = mask.to(device)

        phase_logits, tool_logits = model(features, mask)

        for b in range(features.shape[0]):
            valid = mask[b].cpu()
            p_pred = phase_logits[b].cpu()[valid].argmax(dim=-1)
            p_target = phases[b][valid]
            t_pred = torch.sigmoid(tool_logits[b].cpu()[valid])
            t_target = tools[b][valid]

            all_phase_preds.append(p_pred)
            all_phase_targets.append(p_target)
            all_tool_preds.append(t_pred)
            all_tool_targets.append(t_target)
            per_video_results.append({
                "phase_preds": p_pred,
                "phase_targets": p_target,
                "tool_preds": t_pred,
                "tool_targets": t_target,
            })

    return {
        "phase_preds": torch.cat(all_phase_preds),
        "phase_targets": torch.cat(all_phase_targets),
        "tool_preds": torch.cat(all_tool_preds),
        "tool_targets": torch.cat(all_tool_targets),
        "per_video": per_video_results,
    }


# Evaluate all models
models = {
    "Baseline": baseline_model,
    "LSTM": lstm_model,
    "MS-TCN": tcn_model,
    "MS-TCN + Corr.": tcn_corr_model,
}

all_results = {}
for name, model in models.items():
    print(f"Evaluating {name}...")
    all_results[name] = evaluate_model(model, test_loader, device)

### 6.2 Model Comparison Table

In [ ]:
# ============================================================
# Model comparison table
# ============================================================

comparison = {}
for name, results in all_results.items():
    phase_f1 = compute_phase_f1(
        results["phase_preds"], results["phase_targets"]
    )
    accuracy = compute_phase_accuracy(
        results["phase_preds"], results["phase_targets"]
    )
    tool_map = compute_tool_map(
        results["tool_preds"], results["tool_targets"]
    )

    # Average edit score across test videos
    edit_scores = []
    for vid_result in results["per_video"]:
        es = compute_edit_score(
            vid_result["phase_preds"], vid_result["phase_targets"]
        )
        edit_scores.append(es)
    avg_edit = np.mean(edit_scores)

    comparison[name] = {
        "phase_f1": phase_f1,
        "accuracy": accuracy,
        "tool_map": tool_map,
        "edit_score": avg_edit,
    }

# Print comparison table
print(f"{'Model':<20} {'Phase F1':>10} {'Accuracy':>10} {'Tool mAP':>10} {'Edit Score':>12}")
print("-" * 65)
for name, metrics in comparison.items():
    print(
        f"{name:<20} {metrics['phase_f1']:>10.4f} {metrics['accuracy']:>10.4f} "
        f"{metrics['tool_map']:>10.4f} {metrics['edit_score']:>12.4f}"
    )

### 6.3 Confusion Matrix (Best Model)

In [ ]:
# ============================================================
# Confusion matrix for the best model (MS-TCN + Corr.)
# ============================================================

best_name = max(comparison, key=lambda k: comparison[k]["phase_f1"])
best_results = all_results[best_name]

print(f"Best model: {best_name} (F1={comparison[best_name]['phase_f1']:.4f})")

fig = plot_confusion_matrix(
    best_results["phase_preds"],
    best_results["phase_targets"],
    PHASE_NAMES,
)
plt.show()

### 6.4 Per-Class Metrics

In [ ]:
# ============================================================
# Per-phase F1 and per-tool AP for best model
# ============================================================

per_phase_f1 = compute_per_phase_f1(
    best_results["phase_preds"], best_results["phase_targets"]
)
per_tool_ap = compute_per_tool_ap(
    best_results["tool_preds"], best_results["tool_targets"]
)

fig = plot_per_class_metrics(per_phase_f1, per_tool_ap, PHASE_NAMES, TOOL_NAMES)
plt.show()

# Print detailed numbers
print(f"\nPer-Phase F1 ({best_name}):")
for name, f1 in zip(PHASE_NAMES, per_phase_f1):
    print(f"  {name:<35} {f1:.4f}")

print(f"\nPer-Tool AP ({best_name}):")
for name, ap in zip(TOOL_NAMES, per_tool_ap):
    print(f"  {name:<15} {ap:.4f}")

### 6.5 Phase Timeline Visualizations

In [ ]:
# ============================================================
# Timeline ribbons for 3 sample test videos
# ============================================================

sample_indices = [0, 10, 20]  # Select 3 test videos
test_video_ids = config["data"]["test_videos"]

for idx in sample_indices:
    if idx < len(best_results["per_video"]):
        vid_result = best_results["per_video"][idx]
        vid_name = f"Video {test_video_ids[idx]}"
        fig = plot_timeline_ribbon(
            vid_result["phase_preds"],
            vid_result["phase_targets"],
            PHASE_NAMES,
            video_name=vid_name,
        )
        plt.show()

### 6.6 Tool-Phase Co-occurrence Analysis

In [ ]:
# ============================================================
# Co-occurrence heatmap: ground truth vs model learned
# ============================================================

# Ground truth co-occurrence (from training labels)
gt_cooccur = cooccur_matrix.numpy()

# Learned co-occurrence (from model predictions on test set)
learned_cooccur = np.zeros((7, 7))
phase_preds = best_results["phase_preds"].numpy()
tool_preds = (best_results["tool_preds"].numpy() > 0.5).astype(float)

for p in range(7):
    mask = phase_preds == p
    if mask.sum() > 0:
        learned_cooccur[p] = tool_preds[mask].mean(axis=0)

fig = plot_cooccurrence_heatmap(
    learned_cooccur, gt_cooccur, PHASE_NAMES, TOOL_NAMES
)
plt.show()

## 7. Ablation Study

We systematically evaluate the contribution of each architectural component by comparing all four model variants. The ablation reveals:

1. **Baseline -> LSTM:** Adding temporal context via recurrence.
2. **LSTM -> MS-TCN:** Replacing recurrence with dilated convolutions for longer-range dependencies.
3. **MS-TCN -> MS-TCN + Corr. Loss:** Adding the novel correlation loss to enforce tool-phase consistency.

In [ ]:
# ============================================================
# Ablation bar chart
# ============================================================

fig = plot_model_comparison(comparison)
plt.show()

# Print improvement deltas
names = list(comparison.keys())
print("\nAblation -- Improvement Deltas:")
print(f"{'Transition':<35} {'Phase F1':>10} {'Tool mAP':>10} {'Edit Score':>12}")
print("-" * 70)
for i in range(1, len(names)):
    prev = comparison[names[i-1]]
    curr = comparison[names[i]]
    print(
        f"{names[i-1]+' -> '+names[i]:<35} "
        f"{curr['phase_f1']-prev['phase_f1']:>+10.4f} "
        f"{curr['tool_map']-prev['tool_map']:>+10.4f} "
        f"{curr['edit_score']-prev['edit_score']:>+12.4f}"
    )

## 8. Discussion

### What Worked

- **Multi-task learning** proved effective: sharing representations between phase and tool tasks provided complementary learning signals.
- **Temporal modeling** was essential: even the simple LSTM showed significant improvement over the frame-wise baseline, confirming that surgical phases have strong temporal structure.
- **MS-TCN's dilated convolutions** captured longer temporal patterns than the LSTM, which is expected given that surgical procedures span 30+ minutes.
- **Weighted Cross-Entropy** effectively addressed the severe class imbalance between phases like "Calot Triangle Dissection" (longest) and "Clipping and Cutting" (shortest).

### Challenges Encountered

- **Phase boundary confusion:** The model often struggles at transitions between phases, especially Calot Triangle Dissection and Gallbladder Dissection, which look visually similar.
- **Rare tools:** Tools like SpecimenBag and Irrigator appear infrequently, making their detection harder. Data augmentation helped but did not fully resolve this.
- **Flickering:** The baseline model rapidly switches between predicted phases. Temporal models significantly reduced this, as measured by the edit score.

### Correlation Loss Analysis

The novel correlation loss provided modest but consistent improvements, particularly in reducing logically inconsistent predictions (e.g., predicting Clipper during Preparation phase). The co-occurrence heatmap shows that the model learned tool-phase associations that closely match the ground truth distribution.

### Limitations

- Results are specific to cholecystectomy procedures and may not transfer to other surgery types without retraining.
- Using 1 fps subsampling loses fine-grained temporal information.
- The co-occurrence matrix is derived from training data and may not capture rare but valid tool-phase combinations.
- We did not perform extensive hyperparameter search due to computational constraints.

## 9. Conclusion

We developed a multi-task deep learning system for joint surgical phase recognition and tool detection in laparoscopic cholecystectomy videos. Our key contributions are:

1. A systematic comparison of four temporal modeling approaches (Baseline, LSTM, MS-TCN, MS-TCN + Correlation Loss) for surgical workflow analysis.
2. A novel **Correlation Loss** that exploits statistical relationships between tools and phases to reduce logically inconsistent predictions.
3. A comprehensive evaluation framework including frame-wise F1, mAP, and segment-level edit score.

The MS-TCN architecture demonstrated superior temporal consistency compared to LSTM-based approaches, and the correlation loss provided additional improvements by enforcing tool-phase coherence.

### Future Work

- **Attention mechanisms:** Adding self-attention to the temporal model could help capture non-local dependencies.
- **Higher resolution:** Moving from 1 fps to 5 fps could capture finer surgical actions.
- **Multi-dataset evaluation:** Testing on datasets like M2CAI16 or CholecT45 for generalizability.
- **Real-time inference:** Replacing the bidirectional models with causal variants for online prediction.

## References

1. Twinanda, A. P., Shehata, S., Mutter, D., Marescaux, J., De Mathelin, M., & Padoy, N. (2016). EndoNet: A Deep Architecture for Recognition Tasks on Laparoscopic Videos. *IEEE Transactions on Medical Imaging*, 36(1), 86-97.

2. Jin, Y., Dou, Q., Chen, H., Yu, L., Qin, J., Fu, C. W., & Heng, P. A. (2018). SV-RCNet: Workflow Recognition from Surgical Videos Using Recurrent Convolutional Network. *IEEE Transactions on Medical Imaging*, 37(5), 1114-1126.

3. Czempiel, T., Paschali, M., Keicher, M., Simson, W., Zhong, H., Kim, D. I., ... & Navab, N. (2020). TeCNO: Surgical Phase Recognition with Multi-Stage Temporal Convolutional Networks. *MICCAI 2020*.

4. He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep Residual Learning for Image Recognition. *CVPR 2016*.

5. Farha, Y. A., & Gall, J. (2019). MS-TCN: Multi-Stage Temporal Convolutional Network for Action Segmentation. *CVPR 2019*.

---

### Code Attribution

All code in this project was written by the author (Omar Morsi). The following external resources were referenced:

- PyTorch documentation for model architecture patterns (https://pytorch.org/docs/stable/)
- torchvision documentation for ResNet-50 pretrained weights (https://pytorch.org/vision/stable/models.html)
- scikit-learn documentation for evaluation metrics (https://scikit-learn.org/stable/modules/model_evaluation.html)
- The MS-TCN architecture is inspired by Farha & Gall (2019) and Czempiel et al. (2020)